# add columns: CategoryName or categoryName , Location/lat or location/lat, Location/lng or location/lng ,Date or publishedAtDate 
# apply on Tiktok data
# apply on Youtube data

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os, glob, re, pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,f1_score
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau

2026-05-21 14:13:20.115467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779372800.499846      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779372800.614112      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779372801.627082      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779372801.627143      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779372801.627146      57 computation_placer.cc:177] computation placer alr

In [2]:

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

# لو تعمل داخل Jupyter
from IPython.display import display

In [3]:
# =========================================================
# (0) PATH + SETTINGS  (ALL REGIONS) + QUICK STRUCTURE CHECK
# =========================================================
ALL_REGIONS_ROOT = "/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Google Maps/"
TEXT_COL = "Text_TR"  # ✅ العمود المراد العمل عليه
STAR_CANDIDATES = {"stars", "Stars"}
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir:", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = sorted([d for d in glob.glob(os.path.join(ALL_REGIONS_ROOT, "*")) if os.path.isdir(d)])
print("Region folders found:", len(region_dirs))
print("Regions:", [os.path.basename(d) for d in region_dirs])

# عرض سريع لأول 3 مناطق: عدد المدن داخل كل منطقة
for rd in region_dirs[:3]:
    city_dirs = sorted([d for d in glob.glob(os.path.join(rd, "*")) if os.path.isdir(d)])
    print(f"  - {os.path.basename(rd)}: cities={len(city_dirs)} (sample: {[os.path.basename(x) for x in city_dirs[:5]]})")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir: True
Region folders found: 5
Regions: ['( المنطقة الجنوبية ) Google Maps Data -  After Cleaning', '( المنطقة الشرقية ) Google Maps Data -  After Cleaning', '( المنطقة الشمالية ) Google Maps Data -  After Cleaning', '( المنطقة الغربية ) Google Maps Data -  After Cleaning', '( المنطقة الوسطى ) Google Maps Data -  After Cleaning']
  - ( المنطقة الجنوبية ) Google Maps Data -  After Cleaning: cities=4 (sample: ['منطقة الباحة', 'منطقة جازان', 'منطقة عسير', 'منطقة نجران'])
  - ( المنطقة الشرقية ) Google Maps Data -  After Cleaning: cities=6 (sample: ['الاحساء', 'الجبيل', 'الخبر', 'الدمام', 'الظهران'])
  - ( المنطقة الشمالية ) Google Maps Data -  After Cleaning: cities=5 (sample: ['تبوك', 'محافظة العلا', 'منطقة الجوف', 'منطقة الحدود الشمالية - عرعر', 'منطقة حائل'])


In [4]:
# =========================================================
# Helpers: extract region name (inside parentheses) + city
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def extract_region_from_folder(region_folder_name: str) -> str:
    """
    يستخرج النص بين ( ) من اسم مجلد المنطقة.
    مثال: "( المنطقة الغربية ) Google Maps Data - After Cleaning" -> "المنطقة الغربية"
    إذا لم توجد أقواس يرجع اسم المجلد نفسه.
    """
    m = PAREN_RE.search(region_folder_name)
    if m:
        return m.group(1).strip()
    return region_folder_name.strip()

def list_region_folders(root_folder: str):
    region_dirs = sorted([d for d in glob.glob(os.path.join(root_folder, "*")) if os.path.isdir(d)])
    return region_dirs

def list_city_folders(region_dir: str):
    return sorted([d for d in glob.glob(os.path.join(region_dir, "*")) if os.path.isdir(d)])

def list_files_in_city(city_dir: str):
    return (
        glob.glob(os.path.join(city_dir, "*.xlsx")) +
        glob.glob(os.path.join(city_dir, "*.xls")) +
        glob.glob(os.path.join(city_dir, "*.csv"))
    )

def read_any_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif ext == ".csv":
        # ترميزات عربية شائعة
        try:
            df = pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="cp1256")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_star_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    - يقبل Stars / stars / ... ويوحّدها إلى عمود اسمه 'Stars' إن وجد.
    - لا يغير القيم الآن.
    """
    colmap = {c.lower(): c for c in df.columns}
    # إذا موجود Stars بالاسم الصحيح خلاص
    if "Stars" in df.columns:
        return df

    # ابحث عن أي عمود اسمه stars case-insensitive
    if "stars" in colmap:
        df = df.rename(columns={colmap["stars"]: "Stars"})
        return df

    # مرونة إضافية: إذا فيه rating مثلا
    for cand in list(STAR_CANDIDATES):
        key = cand.lower()
        if key in colmap:
            df = df.rename(columns={colmap[key]: "Stars"})
            return df

    return df  # لم نجد عمود نجوم

In [5]:

# =========================================================
# [STEP 0] Path checks + show regions/cities counts
# =========================================================
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir :", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = list_region_folders(ALL_REGIONS_ROOT)
print("Region folders found:", len(region_dirs))

# عرض أسماء المناطق بصيغة الاسم داخل الأقواس
for rd in region_dirs:
    folder_name = os.path.basename(rd)
    region_name = extract_region_from_folder(folder_name)
    print(f"- Folder: {folder_name}  --> Region: {region_name}")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir : True
Region folders found: 5
- Folder: ( المنطقة الجنوبية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الجنوبية
- Folder: ( المنطقة الشرقية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الشرقية
- Folder: ( المنطقة الشمالية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الشمالية
- Folder: ( المنطقة الغربية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الغربية
- Folder: ( المنطقة الوسطى ) Google Maps Data -  After Cleaning  --> Region: المنطقة الوسطى


In [6]:

# =========================================================
# (0) NORMALIZE COLUMNS
# =========================================================
def normalize_columns(df):
    """
    توحيد أسماء الأعمدة المختلفة لنفس المعنى
    """
    col_map = {
        "CategoryName": ["CategoryName", "categoryName"],
        "Lat": ["Location/lat", "location/lat", "lat", "latitude"],
        "Lng": ["Location/lng", "location/lng", "lng", "longitude"],
        "Date": ["Date", "publishedAtDate", "date"]
    }

    for new_col, possible_cols in col_map.items():
        for c in possible_cols:
            if c in df.columns:
                df[new_col] = df[c]
                break

    return df


# =========================================================
# (1) SCAN AVAILABILITY
# =========================================================
def scan_availability(root_folder: str):
    rows = []
    for rd in list_region_folders(root_folder):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        city_dirs = list_city_folders(rd)
        n_cities = len(city_dirs)
        files = []
        for cd in city_dirs:
            files.extend(list_files_in_city(cd))

        rows.append({
            "Region_Folder": region_folder,
            "Region_Name": region_name,
            "Cities": n_cities,
            "Files": len(files),
            "Sample_File": files[0] if files else None
        })
    return pd.DataFrame(rows).sort_values("Files", ascending=False).reset_index(drop=True)


# =========================================================
# (2) READ ALL REGIONS (CHUNKED)
# =========================================================
def read_all_regions_chunked(root_folder: str, max_files_per_region=None, keep_columns=None):

    all_frames = []
    report_rows = []

    region_dirs = list_region_folders(root_folder)

    for r_idx, rd in enumerate(region_dirs, 1):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)
        city_dirs = list_city_folders(rd)

        region_files = []
        for cd in city_dirs:
            region_files.extend(list_files_in_city(cd))

        if max_files_per_region is not None:
            region_files = region_files[:max_files_per_region]

        print(f"\n[STEP 1] Region {r_idx}/{len(region_dirs)}: {region_name} | cities={len(city_dirs)} | files={len(region_files)}")

        n_rows_region = 0
        has_text = 0
        has_stars = 0
        has_category = 0
        has_lat = 0
        has_lng = 0
        has_date = 0
        read_ok = 0
        read_err = 0

        for i, p in enumerate(region_files, 1):
            try:
                df = read_any_file(p)
                df = standardize_star_column(df)
                df = normalize_columns(df)

                city_name = os.path.basename(os.path.dirname(p))

                df["Region_Folder"] = region_folder
                df["Region_Name"] = region_name
                df["City_Folder"] = city_name
                df["Source_File"] = os.path.basename(p)
                df["__path__"] = p

                # تحقق من توفر الأعمدة
                if TEXT_COL in df.columns:
                    has_text += 1
                if "Stars" in df.columns:
                    has_stars += 1
                if "CategoryName" in df.columns:
                    has_category += 1
                if "Lat" in df.columns:
                    has_lat += 1
                if "Lng" in df.columns:
                    has_lng += 1
                if "Date" in df.columns:
                    has_date += 1

                # تقليل الأعمدة (اختياري)
                if keep_columns is not None:
                    keep = [c for c in keep_columns if c in df.columns]
                    meta = ["Region_Folder","Region_Name","City_Folder","Source_File","__path__"]
                    keep = list(dict.fromkeys(keep + meta))
                    df = df[keep].copy()

                n_rows_region += len(df)
                all_frames.append(df)
                read_ok += 1

                if i <= 2:
                    print(f"  sample file [{i}] {region_name}/{city_name}/{os.path.basename(p)} rows={len(df)}")

                if i % 50 == 0:
                    print(f"  progress: {i}/{len(region_files)} files")

            except Exception as e:
                read_err += 1

        report_rows.append({
            "Region_Name": region_name,
            "Region_Folder": region_folder,
            "Cities": len(city_dirs),
            "Files_Read_OK": read_ok,
            "Files_Read_Err": read_err,
            "Files_with_Text_TR": has_text,
            "Files_with_Stars": has_stars,
            "Files_with_Category": has_category,
            "Files_with_Lat": has_lat,
            "Files_with_Lng": has_lng,
            "Files_with_Date": has_date,
            "Total_Rows_This_Region": n_rows_region
        })

    df_region_report = pd.DataFrame(report_rows) \
        .sort_values("Total_Rows_This_Region", ascending=False) \
        .reset_index(drop=True)

    df_all = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

    return df_all, df_region_report


# =========================================================
# (3) RUN
# =========================================================

print("\n[STEP 1A] Availability (folders/files) per region:")
df_av = scan_availability(ALL_REGIONS_ROOT)
display(df_av)

KEEP_COLS = [
    TEXT_COL,
    "Stars",
    "CategoryName",
    "Lat",
    "Lng",
    "Date"
]

df1, df_region_report = read_all_regions_chunked(
    ALL_REGIONS_ROOT,
    max_files_per_region=None,
    keep_columns=KEEP_COLS
)

print("\n[STEP 1B] Column availability per region (after scan/read):")
display(df_region_report)

print("\n[STEP 1 RESULT] df1 shape:", df1.shape)
print("Columns:", df1.columns.tolist())


[STEP 1A] Availability (folders/files) per region:


,Region_Folder,Region_Name,Cities,Files,Sample_File
0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,4,141,/kaggle/input/datasets/aymanalzahrani7/graduat...
1,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,2,85,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,( المنطقة الغربية ) Google Maps Data - After ...,المنطقة الغربية,6,85,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,( المنطقة الشرقية ) Google Maps Data - After ...,المنطقة الشرقية,6,74,/kaggle/input/datasets/aymanalzahrani7/graduat...
4,( المنطقة الشمالية ) Google Maps Data - After...,المنطقة الشمالية,5,57,/kaggle/input/datasets/aymanalzahrani7/graduat...



[STEP 1] Region 1/5: المنطقة الجنوبية | cities=4 | files=141
  sample file [1] المنطقة الجنوبية/منطقة الباحة/حديقة الفراشة بالمندق_textready.xlsx rows=2715
  sample file [2] المنطقة الجنوبية/منطقة الباحة/أكواخ سار الريفية_textready.xlsx rows=440
  progress: 50/141 files
  progress: 100/141 files

[STEP 1] Region 2/5: المنطقة الشرقية | cities=6 | files=74
  sample file [1] المنطقة الشرقية/الاحساء/قصر محيرس الاثري_textready.xlsx rows=4489
  sample file [2] المنطقة الشرقية/الاحساء/مقهى السيد_textready.xlsx rows=1376
  progress: 50/74 files

[STEP 1] Region 3/5: المنطقة الشمالية | cities=5 | files=57
  sample file [1] المنطقة الشمالية/تبوك/منتجع فيروز_final_final_textready.xlsx rows=425
  sample file [2] المنطقة الشمالية/تبوك/تبوك بارك_final_final_textready.xlsx rows=7242
  progress: 50/57 files

[STEP 1] Region 4/5: المنطقة الغربية | cities=6 | files=85
  sample file [1] المنطقة الغربية/الشعيبة/بحر الشعيبة منطقة السباحة _final_textready.xlsx rows=622
  sample file [2] المنطقة الغربية/الش

,Region_Name,Region_Folder,Cities,Files_Read_OK,Files_Read_Err,Files_with_Text_TR,Files_with_Stars,Files_with_Category,Files_with_Lat,Files_with_Lng,Files_with_Date,Total_Rows_This_Region
0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,4,141,0,141,140,140,140,140,140,276473
1,المنطقة الشرقية,( المنطقة الشرقية ) Google Maps Data - After ...,6,74,0,74,74,74,74,74,74,255183
2,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,2,85,0,85,85,85,85,85,85,226035
3,المنطقة الغربية,( المنطقة الغربية ) Google Maps Data - After ...,6,85,0,85,85,85,85,85,85,186830
4,المنطقة الشمالية,( المنطقة الشمالية ) Google Maps Data - After...,5,57,0,57,57,57,57,57,57,72074



[STEP 1 RESULT] df1 shape: (1016595, 11)
Columns: ['Text_TR', 'Stars', 'CategoryName', 'Lat', 'Lng', 'Date', 'Region_Folder', 'Region_Name', 'City_Folder', 'Source_File', '__path__']


In [7]:
df1['CategoryName'].unique()

array(['حديقة مجتمعية', 'فندق منتجع', 'حديقة', 'مزار سياحي',
       'المحافظة على التراث', 'متنزه حياة برية', 'موقع تاريخي', 'متنزه',
       'متنزه وطني', 'متحف', 'متنزه عام', 'قمة جبل', 'تأجير كوخ',
       'متحف أثري', 'مزرعة', 'فندق', 'الجناح الشاطئي',
       'محمية الحياة البرية', 'مركز تسوق', 'جزيرة', 'مَعلم تاريخي', nan,
       'قلعة', 'موانئ', 'مدينة الملاهي', 'إدارة السياحة البلدية', 'شاطئ',
       'بحيرة السباحة', 'Ford', 'متحف التاريخ الطبيعي', 'سياحة',
       'منطقة مخصصة للسير الطويل', 'خزان مائي', 'كلية فنون', 'مقهى',
       'محمية وطنية', 'سكن', 'مهرجان', 'وكالة للخدمات الترفيهية',
       'Castle', 'National reserve', 'Park', 'Archaeological museum',
       'Market', 'حديقة نباتات', 'Garden', 'Historical landmark', 'سوق',
       'المنزل الريفي', 'الحرف اليدوية', 'بحيرة', 'ميناء',
       'مركز أنشطة تجارية', 'متجر مستلزمات الترفيه على الشواطئ',
       'متنزه المدينة', 'ساحة المأكولات', 'مدينة ملاهي', 'خليج',
       'مدينة ملاهي مائية', 'مركز ثقافي', 'مطعم', 'كافيه', 'هوستيل

In [8]:
 df1['Stars'].value_counts()

Stars
5.0    646067
4.0    166508
3.0    103035
1.0     65871
2.0     34734
Name: count, dtype: int64

In [9]:
df1.head(10)

,Text_TR,Stars,CategoryName,Lat,Lng,Date,Region_Folder,Region_Name,City_Folder,Source_File,__path__
0,جميلة جدا,5.0,حديقة مجتمعية,20.18006,41.26938,2025-10-10T03:30:41.970Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
1,تحتاج صيانه,3.0,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,مكان جميل وشعبي,5.0,حديقة مجتمعية,20.18006,41.26938,2025-09-29T14:32:57.361Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
4,NaN,4.0,حديقة مجتمعية,20.18006,41.26938,2025-09-29T04:15:40.712Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
5,NaN,4.0,حديقة مجتمعية,20.18006,41.26938,2025-09-17T15:37:38.844Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
6,NaN,5.0,حديقة مجتمعية,20.18006,41.26938,2025-09-10T09:32:13.754Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
7,NaN,4.0,حديقة مجتمعية,20.18006,41.26938,2025-09-08T20:46:09.878Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
8,NaN,5.0,حديقة مجتمعية,20.18006,41.26938,2025-09-08T00:45:58.618Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
9,NaN,5.0,حديقة مجتمعية,20.18006,41.26938,2025-09-06T18:30:32.743Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...


In [10]:
# =========================================================
# (2) FILTER ONLY TRULY-EMPTY Text_TR (NaN + "" + "nan" + "[]" ...)
# =========================================================
print(f"\n[STEP 2] Using {TEXT_COL} + strong empty filter")

# ✅ الداتا الجديدة
if TEXT_COL not in df1.columns:
    raise ValueError(f"عمود {TEXT_COL} غير موجود في البيانات.")

before2 = len(df1)

# لا نحول إلى string الآن (مهم)
s = df1[TEXT_COL]

empty_like = {
    "", "nan", "NaN", "none", "None", "NONE",
    "<NA>", "[]", "[ ]", "{}", "null", "NULL"
}

empty_mask = (
    s.isna() |
    s.astype(str).str.strip().isin(empty_like)
)

empty_count = int(empty_mask.sum())

# ✅ الناتج الجديد
df2 = df1.loc[~empty_mask].copy()

# العمود الذي سيستخدم لاحقاً للمودل
df2["TEXT_FOR_MODEL"] = df2[TEXT_COL].astype(str).str.strip()

print("\n[STEP 2 RESULT]")
print("Rows before:", before2)
print("Empty-like rows:", empty_count)
print("Rows after:", len(df2))

print("\nSample empty-like values:")
print(s.loc[empty_mask].head(10).astype(str).tolist())


[STEP 2] Using Text_TR + strong empty filter

[STEP 2 RESULT]
Rows before: 1016595
Empty-like rows: 505054
Rows after: 511541

Sample empty-like values:
['nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan']


In [11]:
df2

,Text_TR,Stars,CategoryName,Lat,Lng,Date,Region_Folder,Region_Name,City_Folder,Source_File,__path__,TEXT_FOR_MODEL
0,جميلة جدا,5.0,حديقة مجتمعية,20.180060,41.269380,2025-10-10T03:30:41.970Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,جميلة جدا
1,تحتاج صيانه,3.0,حديقة مجتمعية,20.180060,41.269380,2025-10-03T11:09:46.968Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,تحتاج صيانه
2,مكان جميل وشعبي,5.0,حديقة مجتمعية,20.180060,41.269380,2025-09-29T14:32:57.361Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,مكان جميل وشعبي
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,حديقة مجتمعية,20.180060,41.269380,2025-09-29T12:19:16.039Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...
16,المكان جميل,5.0,حديقة مجتمعية,20.180060,41.269380,2025-08-29T03:02:49.821Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,المكان جميل
...,...,...,...,...,...,...,...,...,...,...,...,...
1016587,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,1.0,متنزه,26.302181,43.836864,2014-09-19T15:46:09.021Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...
1016588,اكثر من رائعه وجميله,5.0,متنزه,26.302181,43.836864,2014-08-06T11:48:20.060Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,اكثر من رائعه وجميله
1016590,حديقه جدا محترمه,5.0,متنزه,26.302181,43.836864,2014-04-03T10:52:06.492Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقه جدا محترمه
1016593,حديقه حلوه,3.0,متنزه,26.302181,43.836864,2013-07-05T15:31:15.515Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقه حلوه


In [12]:
# =========================================================
# (4) SAFE TEXT CLEANING (INDEPENDENT)
#   - لا يعتمد على Stars_num أو y_bin
#   - يحافظ على الأعمدة التعريفية
# =========================================================

import re
import pandas as pd

print("\n[STEP 4] Safe Cleaning (Independent)")

TEXT_COL = "Text_TR"

if TEXT_COL not in df2.columns:
    raise ValueError(f"{TEXT_COL} غير موجود في df2")

# =========================
# Regex definitions
# =========================
AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
AR_TATWEEL_RE    = re.compile(r"\u0640")

# =========================
# Safe Arabic normalization
# =========================
def normalize_arabic_safe(text: str) -> str:
    text = AR_DIACRITICS_RE.sub("", text)   # remove diacritics
    text = AR_TATWEEL_RE.sub("", text)      # remove tatweel (ـ)
    text = re.sub(r"[إأآا]", "ا", text)      # unify alef only
    return text

# =========================
# Main cleaning function
# =========================
def clean_text(s):
    if pd.isna(s):
        return ""
    
    s = str(s).strip().lower()
    if not s:
        return ""

    # remove links / emails / mentions / hashtags
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # remove emojis
    s = re.sub(r"[\U00010000-\U0010ffff]", " ", s)

    # keep Arabic / English / digits / spaces
    s = re.sub(r"[^0-9a-z\u0600-\u06FF\s]", " ", s)

    # safe normalize
    s = normalize_arabic_safe(s)

    # reduce repeated letters (جمييييل -> جمييل)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove if numbers only
    if re.fullmatch(r"\d+", s):
        return ""

    return s


# =========================
# Preserve metadata columns if exist
# =========================
META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__","CategoryName","Lat","Lng","Date"]
meta_existing = [c for c in META_COLS if c in df2.columns]

# =========================
# BEFORE dataframe
# =========================
df_before_clean = df2[[TEXT_COL] + meta_existing].copy()
df_before_clean = df_before_clean.rename(columns={TEXT_COL: "text_before"})

display(df_before_clean.head(10))


# =========================
# Apply cleaning
# =========================
df4 = df2.copy()
df4["text_clean"] = df4[TEXT_COL].apply(clean_text)

# =========================
# Compare dataframe
# =========================
df_compare = df4[[TEXT_COL, "text_clean"] + meta_existing].copy()
df_compare = df_compare.rename(columns={TEXT_COL: "text_before"})

display(df_compare.head(20))


# =========================
# Removed rows (empty after clean)
# =========================
df_removed = df_compare[df_compare["text_clean"].str.len() == 0].copy()

print("Removed rows (empty after clean):", len(df_removed))
display(df_removed.head(20))


# =========================
# Final cleaned dataframe
# =========================
df4 = df_compare[df_compare["text_clean"].str.len() > 0].copy()

print("Final cleaned shape:", df4.shape)
print("Columns:", df4.columns.tolist())

display(df4.head(20))


# =========================
# Safety check examples
# =========================
print("\n[CHECK] safe normalization examples:")
for t in ["سيئ", "سيئة", "سيء", "المكان سيئ جدا", "المكان رائع جدا", "12345", "مطعم 123"]:
    print(f"{t}  ->  {clean_text(t)}")


[STEP 4] Safe Cleaning (Independent)


,text_before,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
0,جميلة جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-10T03:30:41.970Z
1,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z
2,مكان جميل وشعبي,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T14:32:57.361Z
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z
16,المكان جميل,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-29T03:02:49.821Z
17,اجواء رائعه جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-28T04:45:00.741Z
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-24T17:31:41.972Z
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-21T04:04:17.777Z
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-20T15:52:08.568Z


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
0,جميلة جدا,جميلة جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-10T03:30:41.970Z
1,تحتاج صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T14:32:57.361Z
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z
16,المكان جميل,المكان جميل,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-29T03:02:49.821Z
17,اجواء رائعه جدا,اجواء رائعه جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-28T04:45:00.741Z
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-24T17:31:41.972Z
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-21T04:04:17.777Z
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-20T15:52:08.568Z


Removed rows (empty after clean): 2642


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
24130,٩,,4.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.020606,41.432788,2022-08-16T18:53:34.772Z
30697,٠١,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.020606,41.432788,2021-05-16T18:21:51.165Z
35335,٠,,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.020606,41.432788,2019-08-27T06:27:49.330Z
39763,100,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.020606,41.432788,2019-04-18T21:44:42.376Z
65154,١,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزه غابة خيرة_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.052238,41.397718,2020-09-27T22:33:37.921Z
71100,100,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الأمير محمد بن سعود_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة,20.018918,41.442702,2019-04-18T21:44:25.310Z
80873,٢,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزه الأمير حسام_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,20.011507,41.450208,2021-06-02T14:00:24.319Z
89477,707,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,مزرعة بساتين اللوز - بني ظبيان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,مزرعة,19.994709,41.518766,2024-07-12T15:36:25.558Z
100242,٤,,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة جازان,الكورنيش الجنوبي ( الحزام الجنوبي )_textready....,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,16.876371,42.545444,2021-03-06T10:13:43.459Z
103459,٠,,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة جازان,هايد بارك جازان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,16.907481,42.555279,2022-06-09T14:00:56.309Z


Final cleaned shape: (508899, 12)
Columns: ['text_before', 'text_clean', 'Stars', 'Region_Name', 'Region_Folder', 'City_Folder', 'Source_File', '__path__', 'CategoryName', 'Lat', 'Lng', 'Date']


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
0,جميلة جدا,جميلة جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-10T03:30:41.970Z
1,تحتاج صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T14:32:57.361Z
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z
16,المكان جميل,المكان جميل,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-29T03:02:49.821Z
17,اجواء رائعه جدا,اجواء رائعه جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-28T04:45:00.741Z
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-24T17:31:41.972Z
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-21T04:04:17.777Z
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-20T15:52:08.568Z



[CHECK] safe normalization examples:
سيئ  ->  سيئ
سيئة  ->  سيئة
سيء  ->  سيء
المكان سيئ جدا  ->  المكان سيئ جدا
المكان رائع جدا  ->  المكان رائع جدا
12345  ->  
مطعم 123  ->  مطعم 123


In [13]:
df2

,Text_TR,Stars,CategoryName,Lat,Lng,Date,Region_Folder,Region_Name,City_Folder,Source_File,__path__,TEXT_FOR_MODEL
0,جميلة جدا,5.0,حديقة مجتمعية,20.180060,41.269380,2025-10-10T03:30:41.970Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,جميلة جدا
1,تحتاج صيانه,3.0,حديقة مجتمعية,20.180060,41.269380,2025-10-03T11:09:46.968Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,تحتاج صيانه
2,مكان جميل وشعبي,5.0,حديقة مجتمعية,20.180060,41.269380,2025-09-29T14:32:57.361Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,مكان جميل وشعبي
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,حديقة مجتمعية,20.180060,41.269380,2025-09-29T12:19:16.039Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...
16,المكان جميل,5.0,حديقة مجتمعية,20.180060,41.269380,2025-08-29T03:02:49.821Z,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,المكان جميل
...,...,...,...,...,...,...,...,...,...,...,...,...
1016587,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,1.0,متنزه,26.302181,43.836864,2014-09-19T15:46:09.021Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...
1016588,اكثر من رائعه وجميله,5.0,متنزه,26.302181,43.836864,2014-08-06T11:48:20.060Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,اكثر من رائعه وجميله
1016590,حديقه جدا محترمه,5.0,متنزه,26.302181,43.836864,2014-04-03T10:52:06.492Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقه جدا محترمه
1016593,حديقه حلوه,3.0,متنزه,26.302181,43.836864,2013-07-05T15:31:15.515Z,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقه حلوه


In [14]:
df4

,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
0,جميلة جدا,جميلة جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-10-10T03:30:41.970Z
1,تحتاج صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-10-03T11:09:46.968Z
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-09-29T14:32:57.361Z
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-09-29T12:19:16.039Z
16,المكان جميل,المكان جميل,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-08-29T03:02:49.821Z
...,...,...,...,...,...,...,...,...,...,...,...,...
1016587,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,1.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-09-19T15:46:09.021Z
1016588,اكثر من رائعه وجميله,اكثر من رائعه وجميله,5.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-08-06T11:48:20.060Z
1016590,حديقه جدا محترمه,حديقه جدا محترمه,5.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-04-03T10:52:06.492Z
1016593,حديقه حلوه,حديقه حلوه,3.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2013-07-05T15:31:15.515Z


In [40]:
import re

# =========================
# قاموس الجوانب (Arabic-only) - موسع
# =========================
ASPECT_PATTERNS = {

    # 1) النظافة
    "النظافة": [
        # كلمات مباشرة مع حدود الكلمات
        r"\b(نظافه|نظافة)\b",
        r"\bنظافه\s+(عامه|المكان|المحل|المطعم|الفرع)\b",
        r"\b(نظيف|نظيفه|نظيفة|نظيفين|نظيفون|نظيفات)\b",
        r"\bنظيف(ه)?\s+(جدا|مره|للغايه|للغاية)\b",
        r"\b(متسخ|متسخه|متسخة|متسخين|متسخات)\b",
        r"\b(غير|مو|مش|ما)\s+نظيف(ه)?\b",
        
        # قذارة/وسخ/وصخ/زبالة
        r"\b(وسخ|وسخه|وسخة|وسخان|وسخين|وسخات)\b",
        r"\b(وصخ|وصخه|وصخة|وصخان|وصخين)\b",
        r"\b(قذر|قذره|قذرة|قذرين|قذارات|قذاره|قذارة)\b",
        r"\b(زباله|زبالة|قمامه|قمامة|نفايات|مخلفات)\b",
        r"\bزباله\s+بالارض\b",
        r"\bقمامه\s+موجوده\b",
        
        # تعقيم/روائح مرتبطة بالنظافة
        r"\b(تعقيم|معقم|تطهير|مطهر|منظف|كلور)\b",
        r"\bريح(ه|ة)\s+(وصخه|كريهه|كريهة)\b",
        r"\bريح(ه|ة)\s+مو\s+زينه\b",
        r"\b(روائح\s+كريهه|زفاره|نتن|معفن)\b",
        
        # أرضيات/طاولات/مقاعد
        r"\b(ارضيه|ارضية|ارضيات|الارض|الارضيه|الارضية)\b",
        r"\b(طاولات|الطاولات|كراسي|المقاعد)\s+(وصخه|وسخه|وصخة|وسخة|متسخه|متسخة)\b",
        
        # حشرات كجزء من النظافة
        r"\b(حشره|حشرة|حشرات|نمل|ذباب|صراصير|صرصور)\b"
    ],

    # 2) دورات المياه
    "دورات المياه": [
        r"\bدورات\s+المياه\b",
        r"\bدور(ه|ة)\s+(مياه|المياه|الميه|ميه)\b",
        r"\b(حمام|الحمام|حمامات|دورات|توليت|تواليت|مرحاض|مراحيض)\b",
        
        # توفر/صلاحية
        r"\b(ما|مافي|مو|مش)\s+في\s+(حمام|دورات)\b",
        r"\bالحمام\s+(مقفول|مقفل|خربان)\b",
        r"\b(المرحاض|السيفون|المغسله|المغسلة|مغسله|مغسلة)\s+خربان(ه|ة)?\b",
        
        # مياه/صنابير/مغاسل
        r"\b(مويه|ماء|مياه|حنفيه|حنفية|صنبور|مغسله|مغسلة|مغاسل)\b",
        r"\bمغسل(ه|ة)\s+يدين\b",
        r"\bمكان\s+غسيل\b",
        
        # مستلزمات الحمام
        r"\b(صابون|معقم|مناديل|محارم|منشفه|منشفة|مجفف)\b",
        r"\bورق\s+تواليت\b",
        
        # روائح/اتساخ داخل الحمام
        r"\b(حمام|دورات|الحمامات)\s+(وسخ|وصخ|قذر|متسخ|وسخه|وصخه|قذره|متسخه)\b",
        r"\b(ريح(ه|ة)|زفاره|روائح)\s+(الحمام|كريهه)\b",
        r"\bالحمام\s+ريحته\s+كريهه\b",
        
        # ازدحام/حجم
        r"\b(زحم(ه|ة)|طابور|انتظار)\s+الحمام\b",
        r"\bحمام\s+ضيق\b",
        r"\bحمامات\s+قليله\b"
    ],

    # 3) الخدمة
    "الخدمة": [
        r"\b(خدمه|خدمة|الخدمه|الخدمة)\b",
        r"\bمستوى\s+الخدم(ه|ة)\b",
        r"\b(جود(ه|ة)|نوعي(ه|ة))\s+الخدم(ه|ة)\b",
        r"\bخدم(ه|ة)\s+العملاء\b",
        
        r"\b(تعامل|التعامل|اسلوب|الاسلوب|معامله|معاملة|المعامله|المعاملة)\b",
        r"\bطريق(ه|ة)\s+التعامل\b",
        r"\b(التجاوب|الاهتمام)\b",
        
        r"\b(استقبال|الاستقبال|ترحيب|الترحيب)\b",
        
        r"\b(موظف|موظفين|موظفون|العامل|العمال|الطاقم)\b",
        r"\b(الكاشير|الكاشيره|الكاشيرة|الصراف|المحاسب)\b",
        
        r"\b(مدير|المدير|الاداره|الادارة|المشرف|مسؤول|مسئول)\b",
        
        # صفات إيجابية
        r"\b(متعاون|متعاونين|متعاونون|تعاون|لبق|محترم|احترام)\b",
        r"\b(رايق|را[يى]ق|بشوش|مبتسم)\b",
        
        # صفات سلبية
        r"\b(وقح|وقاحه|وقاحة)\b",
        r"\bقل(ه|ة)\s+ادب\b",
        r"\bعدم\s+احترام\b",
        r"\b(سيء|سوء)\s+التعامل\b",
        r"\bاسلوب\s+سيء\b",
        r"\b(تجاهل|يتجاهل|يتجاهلون)\b",
        
        # الرد والتواصل
        r"\b(ما|مو|مش)\s+يرد(ون)?\b",
        r"\bيردون\s+ببطء\b",
        r"\bرد\s+(متاخر|متأخر|بطيء|بطئ)\b",
        
        # تقييم الخدمة
        r"\bخدم(ه|ة)\s+(سيئه|سيئة|ضعيفه|ضعيفة|ممتازه|ممتازة|رائعه|رائعة|سريعه|سريعة|بطيئه|بطيئة)\b"
    ],

    # 4) مواقف السيارات
    "مواقف السيارات": [
        r"\b(مواقف|موقف)\b",
        r"\bمواقف\s+السيارات\b",
        r"\bموقف\s+(السياره|السيارة|السيارات)\b",
        r"\b(باركنج|بارك[يى]نج)\b",
        
        r"\bمواقف\s+(قليله|قليلة|محدوده|محدودة)\b",
        r"\b(ما|مافي|مو|مش)\s+في\s+مواقف\b",
        r"\bبدون\s+مواقف\b",
        
        r"\bصعب\s+(تلقى|احصل|نلقى)\s+موقف\b",
        r"\b(تدوير|لفات)\s+على\s+موقف\b",
        
        r"\b(زحم(ه|ة)|ازدحام)\s+(مواقف|المواقف)\b",
        r"\bالمواقف\s+(زحمه|زحمة|مكتظه|مكتظة)\b",
        
        r"\bمواقف\s+(بعيده|بعيدة|قريبه|قريبة|مظلمه|مظلمة)\b",
        r"\bمواقف\s+غير\s+(مريحه|مريحة|مرتبه|مرتبة|منظمه|منظمة)\b",
        r"\bمواقف\s+(ترابيه|ترابية)\b",
        r"\bتنظيم\s+المواقف\b",
        
        r"\bمواقف\s+(مخصصه|مخصصة|خاصه|خاصة)\b",
        r"\bمواقف\s+(للعائلات|لذوي\s+الاحتياجات)\b"
    ],

    # 5) الانتظار
    "الانتظار": [
        r"\b(انتظار|الانتظار)\b",
        r"\bوقت\s+انتظار\b",
        r"\b(مد(ه|ة)|فتر(ه|ة))\s+انتظار\b",
        r"\b(طابور|صف|الدور)\b",
        
        r"\b(تاخير|تأخير|تتاخر|تتأخر|يتاخر|يتأخر|متاخر|متأخر)\b",
        
        r"\b(ياخذ|يأخذ)\s+(وقت|وقته)\b",
        r"\b(طول|طويل)\s+وقت\b",
        r"\bوقت\s+طويل\b",
        r"\b(طولنا|تطويل)\b",
        
        r"\b(بطيء|بطئ|بطيئه|بطيئة)\b",
        r"\bبط[يى]ء(\s+جدا)?\b",
        
        r"\b(سرعه|سرعة|سريع|سريعه|سريعة|سريعين|سريعون)\b",
        r"\b(فوري|فورية)\b",
        r"\bبدون\s+انتظار\b",
        
        r"\b(تجهيز|تحضير)\b",
        r"\bتجهيز\s+(بطيء|بطئ|سريع)\b",
        r"\bتاخير\s+التجهيز\b"
    ],

    # 6) الأسعار
    "الأسعار": [
        r"\b(سعر|اسعار|الاسعار|تسعيره|تسعيرة|التسعيره|التسعيرة)\b",
        r"\b(قيمه|قيمة|القيمه|القيمة)\b",
        r"\bقيم(ه|ة)\s+(مقابل|السعر)\b",
        
        r"\b(غالي|غاليه|غالية|غاليين|غاليون)\b",
        r"\bغال[يى](\s+(جدا|مره))?\b",
        r"\b(مرتفع|مرتفعه|مرتفعة)\b",
        
        r"\bمبالغ\s+في(ه|ها)?\b",
        r"\b(مبالغه|مبالغة|استغلال)\b",
        r"\b(ينهبون|ينهب|غلاء)\b",
        
        r"\b(رخيص|رخيصه|رخيصة|رخيصين|رخيصون)\b",
        r"\bرخيص(\s+(جدا|مره))?\b",
        r"\b(مناسب|مناسبه|مناسبة)\b",
        r"\bاسعار\s+مناسب(ه|ة)\b",
        
        r"\b(يستاهل|يستحق)\b",
        r"\b(ما|مو|مش)\s+(يستاهل|يسوى)\b",
        r"\bقيم(ه|ة)\s+(ممتازه|ممتازة|جيده|جيدة)\b",
        
        r"\b(عروض|خصم|تخفيض|تخفيضات)\b",
        r"\bاسعار\s+العروض\b"
    ],

    # 7) الطعام
    "الطعام": [
        r"\b(اكل|الاكل|طعام|الطعام)\b",
        r"\b(وجبه|وجبة|وجبات|صحن|اطباق|طبق)\b",
        
        r"\b(طعم|طعمه|طعمة|مذاق|نكهه|نكهة)\b",
        
        r"\b(لذيذ|لذيذه|لذيذة)\b",
        r"\bلذيذ(\s+(جدا|مره))?\b",
        r"\b(شهي|يشهي|شهية)\b",
        r"\bممتاز\s+الطعم\b",
        
        r"\b(جوده|جودة|الجوده|الجودة)\b",
        r"\bمستوى\s+(الاكل|الطعام)\b",
        
        r"\b(طازج|فريش)\b",
        r"\b(مو|غير|مش)\s+طازج\b",
        r"\b(قديم|بايت)\b",
        
        r"\b(بارد|حار|محروق|يابس|ناشف)\b",
        r"\bحار\s+مره\b",
        r"\bمستوي\s+(زياده|زيادة|ناقص)\b",
        
        r"\b(دهني|دهنية|زيت)\b",
        r"\bزيت\s+كثير\b",
        r"\b(مالح|ملح)\b",
        r"\bملح\s+(زايد|زائد)\b",
        r"\b(حار|سبايسي)\s+(زياده|زيادة)\b",
        

        r"\b(تتبيل|تتبيلة|بهارات|بهار)\b",
        r"\b(صلصه|صلصة|صوص)\b",  # ✅ مع حدود الكلمات
        
        r"\b(غير|مو|مش)\s+لذيذ\b",
        r"\bطعم\s+سيء\b",
        r"\bماله\s+طعم\b"
    ],

    # 8) الإضاءة
    "الإضاءة": [
        r"\b(اضاءه|اضاءة|إضاءه|إضاءة)\b",
        r"\bاضاء(ه|ة)\s+(المكان|ضعيفه|ضعيفة|قويه|قوية|خفيفه|خفيفة|مزعجه|مزعجة)\b",
        r"\bاضاء(ه|ة)\s+(سيئه|سيئة|حلوه|حلوة|جميله|جميلة|ممتازه|ممتازة)\b",
        
        # ⚠️ استثناء كلمة "نور" لوحدها - فقط في سياقات محددة
        r"\b(الانوار|اناره|انارة)\b",  # ✅ بدون "نور" لوحدها
        r"\bانار(ه|ة)\s+(ضعيفه|ضعيفة|قويه|قوية)\b",
        
        r"\b(لمبه|لمبة|لمبات|اللمبات|سبوت|كشاف|كشافات)\b",
        
        r"\b(مظلم|ظلام|معتم)\b",
        r"\bالاضاء(ه|ة)\s+(مطفيه|مطفية|ضعيفه\s+جدا)\b",
        
    ],

    # 9) الزحمة
    "الزحمة": [
        r"\b(زحمه|زحمة|زحام|ازدحام|مزدحم|مكتظ)\b",
        r"\b(كتمه|كتمة|خانقه|خانقة)\b",
        r"\bخانق(ه|ة)\s+مره\b",
        
        r"\b(مكان|الفرع|المحل|المطعم)\s+(زحمه|زحمة)\b",
        
        r"\bازدحام\s+شديد\b",
        r"\bزحم(ه|ة)\s+(شديده|شديدة|قويه|قوية)\b",
        
        r"\b(طاولات|كراسي)\s+(قريبه|قريبة)\b",
        r"\bقريبين\s+من\s+بعض\b",
        r"\bمساح(ه|ة)\s+ضيقه\b",
        r"\b(ضيق|ضيقه|ضيقة)\b",
        
        r"\b(ما|مافي|مو)\s+في\s+جلسات\b",
        r"\bجلسات\s+قليله\b",
        r"\bاماكن\s+قليله\b",
        
        r"\bصعب\s+تلقى\s+مكان\b",
        r"\b(ما|مو)\s+(لقيت|حاصل)\s+مكان\b"
    ],
    
    # 10) الألعاب
    "الألعاب": [
        r"\b(العاب|ألعاب)\b",
        r"\bمنطق(ه|ة)\s+(العاب|ألعاب)\b",
        r"\bقسم\s+(العاب|ألعاب)\b",
        r"\bصال(ه|ة)\s+(العاب|ألعاب)\b",
        
        r"\b(ملاهي|ملاه[يى]|ترفيه|ترفيه[يى])\b",
        r"\b(أركيد|اركيد)\b",
        
        r"\b(بلايستيشن|بلاي\s*ستيشن|سوني)\b",
        r"\b(اكس\s*بوكس|نينتندو)\b",
        
        r"\b(طاول[هة]\s+بلياردو|بلياردو|بولينج|بولنج)\b",
        r"\b(هوكي|اير\s*هوكي)\b",
        
        r"\b(العاب|ألعاب)\s+(اطفال|أطفال|صغار)\b",
        r"\bمنطق(ه|ة)\s+(اطفال|أطفال)\b",
        
        r"\b(زحم(ه|ة)|ازدحام|انتظار|طابور)\s+(العاب|ألعاب)\b",
        
        r"\b(اجهز(ه|ة)|أجهزة|مكائن)\s+(العاب|ألعاب)\b",
        r"\bألعاب\s+الكترونيه\b",
        
        r"\b(تذاكر|كروت|بطاق[هة])\s+(العاب|ألعاب)\b",
        r"\bشحن\s+الكرت\b"
    ],

    # 11) الصيانة
    "الصيانة": [
        r"\b(صيان[هة]|صيانه|صيانة)\b",
        r"\bصيان(ه|ة)\s+(المكان|المحل)\b",
        r"\b(الصيانه|الصيانة|فني|فنيين|فنيون)\b",
        
        r"\b(عطل|اعطال|أعطال)\b",
        r"\b(خربان|خربانه|خربانة|مخرب)\b",
        r"\bخربان(ه|ة)?\s+مره\b",
        r"\b(مو|مش|ما)\s+شغال\b",
        r"\b(لا|ما)\s+(يعمل|يشتغل)\b",
        
        r"\b(تصليح|اصلاح|إصلاح|تصليحات|إصلاحات)\b",
        r"\b(تبديل|تغيير|استبدال)\b",
        
        r"\b(سباك[هة]|سباك|كهرباء|كهربائي|كهربائيه|كهربائية)\b",
        r"\b(تمديدات|تسريب|تهريب)\b",
        
        r"\b(مكيف|التكييف|لمبات|الاضاءه|الاضاءة|مصعد)\s+خربان\b",
        r"\bدور(ه|ة)\s+مياه\s+خربان(ه|ة)\b",
        
        r"\bصيان(ه|ة)\s+ضعيفه\b",
        r"\b(ما|مافي|مو)\s+في\s+صيان(ه|ة)\b",
        r"\b(تاخير|تأخير)\s+الصيان(ه|ة)\b"
    ],
}

# =========================
# Compile regex (important for speed)
# =========================
ASPECT_REGEX = {
    asp: [re.compile(pat) for pat in pats]
    for asp, pats in ASPECT_PATTERNS.items()
}

In [41]:
def detect_aspects_with_hits(text: str):

    if not isinstance(text, str):
        return {}

    found = {}

    for asp, regs in ASPECT_REGEX.items():
        hits = []

        for rg in regs:
            for m in rg.finditer(text):
                hits.append(m.group(0))

        if hits:
            found[asp] = sorted(set(hits))

    return found

df4["aspect_hits"] = df4["text_clean"].apply(detect_aspects_with_hits)

In [42]:
df4

,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date,aspect_hits
0,جميلة جدا,جميلة جدا,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-10-10T03:30:41.970Z,{}
1,تحتاج صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-10-03T11:09:46.968Z,{'الصيانة': ['صيانه']}
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-09-29T14:32:57.361Z,{}
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-09-29T12:19:16.039Z,{'النظافة': ['الارض']}
16,المكان جميل,المكان جميل,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.180060,41.269380,2025-08-29T03:02:49.821Z,{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1016587,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,كانت حديقة جميلة ونظيفه قبل سنة اما الان من اس...,1.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-09-19T15:46:09.021Z,{}
1016588,اكثر من رائعه وجميله,اكثر من رائعه وجميله,5.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-08-06T11:48:20.060Z,{}
1016590,حديقه جدا محترمه,حديقه جدا محترمه,5.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2014-04-03T10:52:06.492Z,{}
1016593,حديقه حلوه,حديقه حلوه,3.0,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,القصيم,منتزه صاحب السمو الملكي الأمير فيصل بن مشعل بن...,/kaggle/input/datasets/aymanalzahrani7/graduat...,متنزه,26.302181,43.836864,2013-07-05T15:31:15.515Z,{}


In [43]:
# ======================================================
# CREATE ASPECT LONG DATAFRAME (ONE ASPECT PER ROW)
# ======================================================

TEXT_COL = "text_clean"   # أو Text_TR

META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__","CategoryName","Lat","Lng","Date"]
meta_existing = [c for c in META_COLS if c in df4.columns]

df4 = df4.copy()

# استخراج الجوانب
df4["aspect_hits"] = df4[TEXT_COL].apply(detect_aspects_with_hits)

# تحويل إلى صف لكل جانب
rows = []

for idx, r in df4.iterrows():

    hits_dict = r["aspect_hits"]

    if not hits_dict:
        continue

    for aspect, hits in hits_dict.items():

        rows.append({
            "row_id": idx,
            "aspect": aspect,
            "hits": " | ".join(hits),
            "text": r[TEXT_COL],
            **{c: r[c] for c in meta_existing}
        })

df_aspect_long = pd.DataFrame(rows)

print("Aspect-level rows:", len(df_aspect_long))
display(df_aspect_long.head(20))

print("\nAspect distribution:")
display(df_aspect_long["aspect"].value_counts())

Aspect-level rows: 184475


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date
0,1,الصيانة,صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z
1,3,النظافة,الارض,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z
2,19,الألعاب,العاب | العاب اطفال,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z
3,19,الصيانة,تغيير,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z
4,27,الأسعار,اسعار,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-20T15:52:08.568Z
5,53,الزحمة,زحمة,زحمة ولا تلقي مكان تجلس فيه,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T14:07:47.766Z
6,54,الإضاءة,اناره,جدا ممتاز و مساحته حلوه و تنوع الفعاليات ممتاز...,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T12:44:28.312Z
7,56,الأسعار,يستحق,المنتزة جميل واسعاره معقوله يستحق الزيارة,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T03:08:25.769Z
8,58,الخدمة,الاهتمام,الموقع ينقصه الاهتمام بالنظافة وزيادة الرقعة ا...,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-10T20:21:20.301Z
9,67,الأسعار,ما يستاهل | يستاهل,ما يستاهل الزيارة,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-07T03:49:18.616Z



Aspect distribution:


aspect
الأسعار           53191
الخدمة            30319
النظافة           20893
الطعام            19983
دورات المياه      16337
الألعاب           14373
الزحمة            10744
مواقف السيارات     6669
الصيانة            5555
الانتظار           4896
الإضاءة            1515
Name: count, dtype: int64

In [19]:
#graph

In [44]:
aspect_overall = (
    df_aspect_long
    .groupby("aspect")
    .size()
    .reset_index(name="mentions")
    .sort_values("mentions", ascending=False)
)

display(aspect_overall)

,aspect,mentions
0,الأسعار,53191
4,الخدمة,30319
8,النظافة,20893
7,الطعام,19983
9,دورات المياه,16337
1,الألعاب,14373
5,الزحمة,10744
10,مواقف السيارات,6669
6,الصيانة,5555
3,الانتظار,4896


In [45]:
total_mentions = aspect_overall["mentions"].sum()

aspect_overall["percentage_%"] = (
    aspect_overall["mentions"] / total_mentions * 100
).round(2)

display(aspect_overall)

,aspect,mentions,percentage_%
0,الأسعار,53191,28.83
4,الخدمة,30319,16.44
8,النظافة,20893,11.33
7,الطعام,19983,10.83
9,دورات المياه,16337,8.86
1,الألعاب,14373,7.79
5,الزحمة,10744,5.82
10,مواقف السيارات,6669,3.62
6,الصيانة,5555,3.01
3,الانتظار,4896,2.65


In [46]:
aspect_unique_reviews = (
    df_aspect_long
    .groupby("aspect")["row_id"]
    .nunique()
    .reset_index(name="unique_reviews")
    .sort_values("unique_reviews", ascending=False)
)

display(aspect_unique_reviews)

,aspect,unique_reviews
0,الأسعار,53191
4,الخدمة,30319
8,النظافة,20893
7,الطعام,19983
9,دورات المياه,16337
1,الألعاب,14373
5,الزحمة,10744
10,مواقف السيارات,6669
6,الصيانة,5555
3,الانتظار,4896


In [47]:
aspect_by_region = (
    df_aspect_long
    .groupby(["Region_Name", "aspect"])
    .size()
    .reset_index(name="mentions")
    .sort_values(["Region_Name","mentions"], ascending=[True, False])
)

display(aspect_by_region.head(30))

,Region_Name,aspect,mentions
0,المنطقة الجنوبية,الأسعار,13917
4,المنطقة الجنوبية,الخدمة,8192
8,المنطقة الجنوبية,النظافة,7044
9,المنطقة الجنوبية,دورات المياه,6923
7,المنطقة الجنوبية,الطعام,4353
1,المنطقة الجنوبية,الألعاب,3913
5,المنطقة الجنوبية,الزحمة,3876
10,المنطقة الجنوبية,مواقف السيارات,2630
6,المنطقة الجنوبية,الصيانة,1666
3,المنطقة الجنوبية,الانتظار,950


In [24]:
#grsph - power bi

In [48]:
region_pivot = aspect_by_region.pivot_table(
    index="Region_Name",
    columns="aspect",
    values="mentions",
    fill_value=0
)

display(region_pivot)

aspect,الأسعار,الألعاب,الإضاءة,الانتظار,الخدمة,الزحمة,الصيانة,الطعام,النظافة,دورات المياه,مواقف السيارات
Region_Name,,,,,,,,,,,
المنطقة الجنوبية,13917.0,3913.0,393.0,950.0,8192.0,3876.0,1666.0,4353.0,7044.0,6923.0,2630.0
المنطقة الشرقية,13036.0,3689.0,296.0,1286.0,6377.0,2486.0,1369.0,5812.0,4555.0,2804.0,1491.0
المنطقة الشمالية,4351.0,782.0,61.0,654.0,4232.0,469.0,417.0,2579.0,1455.0,786.0,301.0
المنطقة الغربية,8027.0,1846.0,307.0,536.0,3191.0,1305.0,792.0,1828.0,3304.0,2324.0,757.0
المنطقة الوسطى,13860.0,4143.0,458.0,1470.0,8327.0,2608.0,1311.0,5411.0,4535.0,3500.0,1490.0


In [49]:
region_pivot = aspect_by_region.pivot_table(
    index="Region_Name",
    columns="aspect",
    values="mentions",
    fill_value=0
)

display(region_pivot)

aspect,الأسعار,الألعاب,الإضاءة,الانتظار,الخدمة,الزحمة,الصيانة,الطعام,النظافة,دورات المياه,مواقف السيارات
Region_Name,,,,,,,,,,,
المنطقة الجنوبية,13917.0,3913.0,393.0,950.0,8192.0,3876.0,1666.0,4353.0,7044.0,6923.0,2630.0
المنطقة الشرقية,13036.0,3689.0,296.0,1286.0,6377.0,2486.0,1369.0,5812.0,4555.0,2804.0,1491.0
المنطقة الشمالية,4351.0,782.0,61.0,654.0,4232.0,469.0,417.0,2579.0,1455.0,786.0,301.0
المنطقة الغربية,8027.0,1846.0,307.0,536.0,3191.0,1305.0,792.0,1828.0,3304.0,2324.0,757.0
المنطقة الوسطى,13860.0,4143.0,458.0,1470.0,8327.0,2608.0,1311.0,5411.0,4535.0,3500.0,1490.0


In [50]:
region_pivot = aspect_by_region.pivot_table(
    index="Region_Name",
    columns="aspect",
    values="mentions",
    fill_value=0
)

display(region_pivot)

aspect,الأسعار,الألعاب,الإضاءة,الانتظار,الخدمة,الزحمة,الصيانة,الطعام,النظافة,دورات المياه,مواقف السيارات
Region_Name,,,,,,,,,,,
المنطقة الجنوبية,13917.0,3913.0,393.0,950.0,8192.0,3876.0,1666.0,4353.0,7044.0,6923.0,2630.0
المنطقة الشرقية,13036.0,3689.0,296.0,1286.0,6377.0,2486.0,1369.0,5812.0,4555.0,2804.0,1491.0
المنطقة الشمالية,4351.0,782.0,61.0,654.0,4232.0,469.0,417.0,2579.0,1455.0,786.0,301.0
المنطقة الغربية,8027.0,1846.0,307.0,536.0,3191.0,1305.0,792.0,1828.0,3304.0,2324.0,757.0
المنطقة الوسطى,13860.0,4143.0,458.0,1470.0,8327.0,2608.0,1311.0,5411.0,4535.0,3500.0,1490.0


In [51]:
aspect_by_city = (
    df_aspect_long
    .groupby(["City_Folder", "aspect"])
    .size()
    .reset_index(name="mentions")
)

display(aspect_by_city.head(20))

,City_Folder,aspect,mentions
0,الاحساء,الأسعار,4195
1,الاحساء,الألعاب,977
2,الاحساء,الإضاءة,147
3,الاحساء,الانتظار,283
4,الاحساء,الخدمة,1595
5,الاحساء,الزحمة,526
6,الاحساء,الصيانة,356
7,الاحساء,الطعام,1959
8,الاحساء,النظافة,1070
9,الاحساء,دورات المياه,836


In [52]:
top_aspects_city = (
    aspect_by_city
    .sort_values(["City_Folder","mentions"], ascending=[True, False])
    .groupby("City_Folder")
    .head(3)
)

display(top_aspects_city)

,City_Folder,aspect,mentions
0,الاحساء,الأسعار,4195
7,الاحساء,الطعام,1959
4,الاحساء,الخدمة,1595
19,الجبيل,النظافة,1288
11,الجبيل,الأسعار,1142
...,...,...,...
234,منطقة عسير,الخدمة,3648
239,منطقة عسير,دورات المياه,2849
241,منطقة نجران,الأسعار,166
249,منطقة نجران,النظافة,156


In [53]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [54]:
# =========================
# CONFIG
# =========================
MODEL_NAME = "/kaggle/input/models/aymanalzahrani7/googlecamelbert-v2/pytorch/default/1/Camel BERT v2"  # مثال شائع
MAX_LEN = 128
BATCH_SIZE = 64   # جرّب 64 على GPU، إذا حصل OOM خفّض لـ 32
SAT_MODE = "pos_only"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# (2) DEVICE + LOAD MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("id2label:", model.config.id2label)

# Identify label indices robustly
id2label = {int(k): v for k, v in model.config.id2label.items()}
label_lower = {i: str(lab).lower() for i, lab in id2label.items()}

def _find_idx(keys):
    for i, lab in label_lower.items():
        if any(k in lab for k in keys):
            return i
    return None

pos_idx = _find_idx(["pos", "positive", "ايجاب", "إيجاب"])
neg_idx = _find_idx(["neg", "negative", "سلب", "سلبي"])

# Fallback if model uses common ordering but labels are generic like LABEL_0/1/2
if pos_idx is None or neg_idx is None:
    
    neg_idx,pos_idx =  0,1

print("Indices -> pos:", pos_idx,"neg:", neg_idx)

Device: cuda
Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id2label: {0: 'Negative', 1: 'Positive'}
Indices -> pos: 1 neg: 0


In [55]:
# =========================
# (3) BUILD BERT INPUT (Aspect-aware)
# =========================
# Each row: aspect [SEP] text
df_aspect_long["bert_input"] = (
    df_aspect_long["aspect"].astype(str) + " [SEP] " + df_aspect_long["text"].astype(str)
)

# Optional: drop empty
df_aspect_long = df_aspect_long[df_aspect_long["bert_input"].str.len() > 0].copy()
df_aspect_long.reset_index(drop=True, inplace=True)

print("After bert_input:", df_aspect_long.shape)


After bert_input: (184475, 15)


In [56]:
# =========================
# (4) BATCH INFERENCE FUNCTION (BINARY)
# =========================
def batch_predict_binary(texts, batch_size=64, max_len=128, log_every_batches=300):
    """
    Returns:
      p_pos, p_neg (np arrays length N)
    """
    all_pos, all_neg = [], []

    N = len(texts)

    for b, start in enumerate(range(0, N, batch_size), 1):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits   # [B, 2]

        probs = F.softmax(logits, dim=1)   # [B, 2]
        probs = probs.detach().cpu().numpy()

        all_pos.append(probs[:, pos_idx])
        all_neg.append(probs[:, neg_idx])

        if (b % log_every_batches) == 0:
            done = min(start + batch_size, N)
            print(f"Processed {done}/{N} rows...")

    p_pos = np.concatenate(all_pos, axis=0)
    p_neg = np.concatenate(all_neg, axis=0)

    return p_pos, p_neg


# =========================
# (5) RUN INFERENCE
# =========================
texts = df_aspect_long["bert_input"].tolist()

p_pos, p_neg = batch_predict_binary(
    texts,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    log_every_batches=300
)

df_aspect_long["p_positive"] = p_pos
df_aspect_long["p_negative"] = p_neg


# Final label by argmax
# stack order = [neg, pos]
pred_idx = np.argmax(
    np.stack([
        df_aspect_long["p_negative"],
        df_aspect_long["p_positive"]
    ], axis=1),
    axis=1
)

# idx 0 = negative, idx 1 = positive
df_aspect_long["sentiment"] = np.where(
    pred_idx == 1,
    "ايجابي",
    "سلبي"
)


# Satisfaction score
# In binary setup, usually positive probability is enough
df_aspect_long["satisfaction_score"] = df_aspect_long["p_positive"]


print("Inference done.")
display(df_aspect_long.head(10))

Processed 19200/184475 rows...
Processed 38400/184475 rows...
Processed 57600/184475 rows...
Processed 76800/184475 rows...
Processed 96000/184475 rows...
Processed 115200/184475 rows...
Processed 134400/184475 rows...
Processed 153600/184475 rows...
Processed 172800/184475 rows...
Inference done.


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,CategoryName,Lat,Lng,Date,bert_input,p_positive,p_negative,sentiment,satisfaction_score
0,1,الصيانة,صيانه,تحتاج صيانه,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-10-03T11:09:46.968Z,الصيانة [SEP] تحتاج صيانه,0.024958,0.975042,سلبي,0.024958
1,3,النظافة,الارض,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-09-29T12:19:16.039Z,النظافة [SEP] رساله اوجهها للي باع لي الطير في...,0.000473,0.999527,سلبي,0.000473
2,19,الألعاب,العاب | العاب اطفال,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z,الألعاب [SEP] منتزة جميل فية العاب اطفال برسوم...,0.000577,0.999423,سلبي,0.000577
3,19,الصيانة,تغيير,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-25T06:14:45.831Z,الصيانة [SEP] منتزة جميل فية العاب اطفال برسوم...,0.000671,0.999329,سلبي,0.000671
4,27,الأسعار,اسعار,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-20T15:52:08.568Z,الأسعار [SEP] مبالغين في اسعار الالعاب في المد...,0.000489,0.999511,سلبي,0.000489
5,53,الزحمة,زحمة,زحمة ولا تلقي مكان تجلس فيه,2.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T14:07:47.766Z,الزحمة [SEP] زحمة ولا تلقي مكان تجلس فيه,0.000856,0.999144,سلبي,0.000856
6,54,الإضاءة,اناره,جدا ممتاز و مساحته حلوه و تنوع الفعاليات ممتاز...,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T12:44:28.312Z,الإضاءة [SEP] جدا ممتاز و مساحته حلوه و تنوع ا...,0.999598,0.000402,ايجابي,0.999598
7,56,الأسعار,يستحق,المنتزة جميل واسعاره معقوله يستحق الزيارة,5.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-11T03:08:25.769Z,الأسعار [SEP] المنتزة جميل واسعاره معقوله يستح...,0.999688,0.000312,ايجابي,0.999688
8,58,الخدمة,الاهتمام,الموقع ينقصه الاهتمام بالنظافة وزيادة الرقعة ا...,3.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-10T20:21:20.301Z,الخدمة [SEP] الموقع ينقصه الاهتمام بالنظافة وز...,0.002974,0.997026,سلبي,0.002974
9,67,الأسعار,ما يستاهل | يستاهل,ما يستاهل الزيارة,1.0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...,حديقة مجتمعية,20.18006,41.26938,2025-08-07T03:49:18.616Z,الأسعار [SEP] ما يستاهل الزيارة,0.001129,0.998871,سلبي,0.001129


In [57]:
# =========================
# (6) SATISFACTION STATS
# =========================

# --- Overall per aspect
aspect_satisfaction = (
    df_aspect_long
    .groupby("aspect")
    .agg(
        mentions=("aspect", "size"),
        unique_reviews=("row_id", "nunique") if "row_id" in df_aspect_long.columns else ("aspect", "size"),
        avg_p_positive=("p_positive", "mean"),
        avg_p_negative=("p_negative", "mean"),
        satisfaction=("satisfaction_score", "mean"),
    )
    .reset_index()
)

aspect_satisfaction["satisfaction_%"] = (aspect_satisfaction["satisfaction"] * 100).round(2)
aspect_satisfaction = aspect_satisfaction.sort_values("satisfaction_%", ascending=True)

print("\n[Overall Aspect Satisfaction]")
display(aspect_satisfaction)



[Overall Aspect Satisfaction]


,aspect,mentions,unique_reviews,avg_p_positive,avg_p_negative,satisfaction,satisfaction_%
6,الصيانة,5555,5555,0.540917,0.459083,0.540917,54.090000
2,الإضاءة,1515,1515,0.633412,0.366588,0.633412,63.340000
3,الانتظار,4896,4896,0.655825,0.344175,0.655825,65.580002
4,الخدمة,30319,30319,0.682118,0.317882,0.682118,68.209999
5,الزحمة,10744,10744,0.711938,0.288062,0.711938,71.190002
0,الأسعار,53191,53191,0.720471,0.279529,0.720471,72.050003
8,النظافة,20893,20893,0.722892,0.277108,0.722892,72.290001
9,دورات المياه,16337,16337,0.723119,0.276881,0.723119,72.309998
10,مواقف السيارات,6669,6669,0.754804,0.245196,0.754804,75.480003
7,الطعام,19983,19983,0.780282,0.219718,0.780282,78.029999


In [58]:

# --- By Region (if exists)
aspect_by_region = None
if "Region_Name" in df_aspect_long.columns:
    aspect_by_region = (
        df_aspect_long
        .groupby(["Region_Name", "aspect"])
        .agg(
            mentions=("aspect", "size"),
            unique_reviews=("row_id", "nunique") if "row_id" in df_aspect_long.columns else ("aspect", "size"),
            satisfaction=("satisfaction_score", "mean"),
            avg_p_positive=("p_positive", "mean"),
            avg_p_negative=("p_negative", "mean"),
        )
        .reset_index()
    )
    aspect_by_region["satisfaction_%"] = (aspect_by_region["satisfaction"] * 100).round(2)
    print("\n[Aspect Satisfaction by Region]")
    display(aspect_by_region.head(30))
else:
    print("\nRegion_Name not found -> skipping region stats.")

# --- By City (if exists)
aspect_by_city = None
if "City_Folder" in df_aspect_long.columns:
    aspect_by_city = (
        df_aspect_long
        .groupby(["City_Folder", "aspect"])
        .agg(
            mentions=("aspect", "size"),
            unique_reviews=("row_id", "nunique") if "row_id" in df_aspect_long.columns else ("aspect", "size"),
            satisfaction=("satisfaction_score", "mean"),
            avg_p_positive=("p_positive", "mean"),
            avg_p_negative=("p_negative", "mean"),
        )
        .reset_index()
    )
    aspect_by_city["satisfaction_%"] = (aspect_by_city["satisfaction"] * 100).round(2)
    print("\n[Aspect Satisfaction by City]")
    display(aspect_by_city.head(30))
else:
    print("\nCity_Folder not found -> skipping city stats.")


[Aspect Satisfaction by Region]


,Region_Name,aspect,mentions,unique_reviews,satisfaction,avg_p_positive,avg_p_negative,satisfaction_%
0,المنطقة الجنوبية,الأسعار,13917,13917,0.698214,0.698214,0.301786,69.820000
1,المنطقة الجنوبية,الألعاب,3913,3913,0.862217,0.862217,0.137783,86.220001
2,المنطقة الجنوبية,الإضاءة,393,393,0.604900,0.604900,0.395100,60.490002
3,المنطقة الجنوبية,الانتظار,950,950,0.600162,0.600162,0.399838,60.020000
4,المنطقة الجنوبية,الخدمة,8192,8192,0.630652,0.630652,0.369348,63.070000
5,المنطقة الجنوبية,الزحمة,3876,3876,0.710070,0.710070,0.289930,71.010002
6,المنطقة الجنوبية,الصيانة,1666,1666,0.562800,0.562800,0.437200,56.279999
7,المنطقة الجنوبية,الطعام,4353,4353,0.795072,0.795072,0.204927,79.510002
8,المنطقة الجنوبية,النظافة,7044,7044,0.692437,0.692437,0.307563,69.239998
9,المنطقة الجنوبية,دورات المياه,6923,6923,0.733708,0.733708,0.266292,73.370003



[Aspect Satisfaction by City]


,City_Folder,aspect,mentions,unique_reviews,satisfaction,avg_p_positive,avg_p_negative,satisfaction_%
0,الاحساء,الأسعار,4195,4195,0.780865,0.780865,0.219135,78.089996
1,الاحساء,الألعاب,977,977,0.895868,0.895868,0.104132,89.589996
2,الاحساء,الإضاءة,147,147,0.673966,0.673966,0.326034,67.400002
3,الاحساء,الانتظار,283,283,0.700457,0.700457,0.299543,70.050003
4,الاحساء,الخدمة,1595,1595,0.644358,0.644358,0.355642,64.440002
5,الاحساء,الزحمة,526,526,0.714483,0.714483,0.285517,71.449997
6,الاحساء,الصيانة,356,356,0.559019,0.559019,0.440981,55.900002
7,الاحساء,الطعام,1959,1959,0.841771,0.841771,0.158229,84.180000
8,الاحساء,النظافة,1070,1070,0.715410,0.715410,0.284590,71.540001
9,الاحساء,دورات المياه,836,836,0.709726,0.709726,0.290274,70.970001


In [59]:
# احفظ التفاصيل (كل صف = جانب داخل تعليق + احتمالات)
df_aspect_long.drop(columns=["bert_input"], errors="ignore").to_csv(
    "/kaggle/working/aspect_sentiment_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

# احفظ ملخص الرضا لكل جانب
aspect_satisfaction.to_csv(
    "/kaggle/working/aspect_satisfaction.csv",
    index=False,
    encoding="utf-8-sig"
)

# إذا عندك إحصائيات حسب المنطقة/المدينة
if "aspect_by_region" in globals() and aspect_by_region is not None:
    aspect_by_region.to_csv(
        "/kaggle/working/aspect_satisfaction_by_region.csv",
        index=False,
        encoding="utf-8-sig"
    )

if "aspect_by_city" in globals() and aspect_by_city is not None:
    aspect_by_city.to_csv(
        "/kaggle/working/aspect_satisfaction_by_city.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("Saved to /kaggle/working/")

Saved to /kaggle/working/
